<a href="https://colab.research.google.com/github/DANFELcode/IAIA-Proyecto-Clue/blob/main/Proyecto_Corte3_Azul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto IAIA

## Equipo Azul

## Daniel Felipe Sua Siempira
## Juan David Moreno D'Aleman

## Clue


## Introducción

Este proyecto aplica los fundamentos de la IA simbólica para resolver un problema concreto mediante un sistema basado en reglas. Modelamos el razonamiento de un jugador de Clue (Cluedo) como un conjunto de proposiciones atómicas y cláusulas definidas de Horn, y lo ejecutamos con el motor de inferencia desarrollado en el laboratorio, que maneja tres estados de conocimiento: YES, NO y UNKNOWN. El sistema actúa como un asesor: a partir de las cartas que el jugador conoce, deduce con forward chaining qué parte de la solución ya está descubierta, y con backward chaining responde si ya puede hacer su acusación final, preguntándole al usuario los datos que falten. Además, explica cada conclusión mostrando la secuencia de reglas que encadenó para llegar a ella.

## ¿De qué trata el juego?

Clue es un juego de mesa de deducción en el que se debe descubrir quién cometió un crimen, con qué arma y en qué habitación. Al comenzar, se guarda en un sobre secreto una carta de cada categoría (un sospechoso, un arma y un lugar), y las demás cartas se reparten entre los jugadores. A lo largo de la partida, los jugadores van conociendo cartas, las propias y las que otros les muestran, y las descartan como posibles soluciones. Gana quien logra identificar correctamente las tres cartas del sobre y hacer la acusación. En nuestra versión reducida hay 4 sospechosos (Coronel Mostaza, Señorita Escarlata, Señora Blanco y Reverendo Verde), 4 armas (Candelabro, Cuchillo, Tubería de plomo y Revólver) y 4 habitaciones (Cocina, Estudio, Biblioteca y Comedor).

## 1. Conceptos básicos del sistema

Antes de ver el modelo, estos son los conceptos que usa el motor y cómo se ven en nuestro proyecto.

| Concepto | Qué es | Ejemplo en Clue |
|---|---|---|
| **Proposición atómica** | Una afirmación que puede ser verdadera o falsa. Tiene un `symbol` (nombre corto) y un `meaning` (frase en lenguaje natural). | `vista_mostaza`: "la carta del Coronel Mostaza está en una mano conocida" |
| **Regla (cláusula de Horn)** | Una implicación con varias condiciones y una sola conclusión: `P1 ∧ P2 ∧ ... ∧ Pn ⇒ Q`. | Si vi a Escarlata, Blanco y Verde, entonces el culpable es Mostaza |
| **Base de conocimiento (KB)** | El conocimiento *estable* del dominio: todas las proposiciones y todas las reglas. No cambia entre partidas. | Las 28 proposiciones y las 25 reglas de este proyecto |
| **Base de hechos (FB)** | El conocimiento *de un caso particular*: qué cartas conoce el jugador ahora mismo y qué se ha deducido. | Las cartas que tengo en mano y las que me mostraron |
| **Forward chaining** | Parte de los hechos conocidos y aplica reglas para descubrir todo lo que se pueda deducir. | De mis cartas vistas deduzco quién es el culpable |
| **Backward chaining** | Parte de una meta y busca qué hace falta para demostrarla; si falta un dato, se lo pregunta al usuario. | Meta: `puedo_acusar`. ¿Qué me falta saber? |
| **Explicación (cómo)** | Muestra la cadena de reglas que llevó a una conclusión. | `puedo_acusar` ← R25 ← R15 ← R3 ← cartas vistas |

### Los tres valores de una proposición

| Valor | Significado | Ejemplo |
|---|---|---|
| `YES` | Se cumple | Sé que la carta de Mostaza está en una mano conocida |
| `NO` | No se cumple | Sé que esa carta no está en una mano conocida |
| `UNKNOWN` | Se sabe que no se sabe: el sistema ya lo intentó y no pudo determinarlo | Nadie me ha mostrado esa carta |

**Diferencia importante:** una proposición *ausente* de la base de hechos significa "todavía no se ha preguntado", y el motor puede preguntársela al usuario. Una proposición con valor `UNKNOWN` significa "ya se intentó", y el motor **no vuelve a preguntar**.

## 2. Diseño del problema y del modelo

### 2.1 ¿Qué problema resolvemos?

Un jugador de Clue necesita saber **cuándo ya tiene suficiente información para acusar** sin arriesgarse. El sistema recibe las cartas que el jugador conoce y responde dos cosas:

1. **¿Qué parte del crimen ya está descubierta?** (forward chaining)
2. **¿Puedo hacer la acusación final?** (backward chaining, preguntando lo que falte)

### 2.2 Reglas del juego en nuestra versión reducida

| Categoría | Cartas (4 por categoría) |
|---|---|
| Sospechosos | Coronel Mostaza, Señorita Escarlata, Señora Blanco, Reverendo Verde |
| Armas | Candelabro, Cuchillo, Tubería de plomo, Revólver |
| Habitaciones | Cocina, Estudio, Biblioteca, Comedor |

- Hay **12 cartas** en total. Tres de ellas, **una por categoría**, están en un sobre secreto: son la solución.
- Las otras 9 cartas están repartidas entre los jugadores.
- El jugador conoce **sus propias cartas** y las que **otros jugadores le muestran**.
- Toda carta conocida queda **descartada** como solución.

### 2.3 Supuestos del modelo

| Supuesto | Justificación |
|---|---|
| En cada categoría hay exactamente una carta en el sobre | Es la regla del juego, y permite concluir por descarte |
| Una carta "vista" es la mía o una que me mostraron (misma proposición) | Para el razonamiento da igual quién la tenga |
| No se modelan contradicciones (por ejemplo, ver las 4 cartas de una categoría) | Los escenarios son controlados y no ocurren |
| No se modelan disyunciones ("me mostró una de estas tres") | Una cláusula de Horn no admite "o" |
| El pasadizo secreto Cocina–Estudio no interviene | Nuestro razonamiento es de descarte de cartas, no de movimiento |

### 2.4 Proposiciones atómicas (28)

Las proposiciones se organizan en **4 grupos**. Los nombres (`symbol`) son semánticos: se leen solos en las explicaciones del sistema.

| Grupo | Cantidad | Quién les da valor | Para qué sirven |
|---|---|---|---|
| 1. Cartas vistas | 12 | El usuario | Es la información de entrada |
| 2. Solución | 12 | Las reglas (deducidas) | Dicen cuál carta es la del sobre |
| 3. Categoría identificada | 3 | Las reglas (deducidas) | Resumen: "esta categoría ya está resuelta" |
| 4. Meta | 1 | Las reglas (deducida) | Responde si ya se puede acusar |

#### Grupo 1: Cartas vistas (entrada)

"Mano conocida" significa la del propio jugador o la de otro jugador que le mostró la carta.

| `symbol` | `meaning` |
|---|---|
| `vista_mostaza` | la carta del Coronel Mostaza está en una mano conocida |
| `vista_escarlata` | la carta de la Señorita Escarlata está en una mano conocida |
| `vista_blanco` | la carta de la Señora Blanco está en una mano conocida |
| `vista_verde` | la carta del Reverendo Verde está en una mano conocida |
| `vista_candelabro` | la carta del Candelabro está en una mano conocida |
| `vista_cuchillo` | la carta del Cuchillo está en una mano conocida |
| `vista_tuberia` | la carta de la Tubería de plomo está en una mano conocida |
| `vista_revolver` | la carta del Revólver está en una mano conocida |
| `vista_cocina` | la carta de la Cocina está en una mano conocida |
| `vista_estudio` | la carta del Estudio está en una mano conocida |
| `vista_biblioteca` | la carta de la Biblioteca está en una mano conocida |
| `vista_comedor` | la carta del Comedor está en una mano conocida |

#### Grupo 2: Solución (deducidas)

| `symbol` | `meaning` |
|---|---|
| `culpable_mostaza` | el culpable es el Coronel Mostaza |
| `culpable_escarlata` | el culpable es la Señorita Escarlata |
| `culpable_blanco` | el culpable es la Señora Blanco |
| `culpable_verde` | el culpable es el Reverendo Verde |
| `arma_candelabro` | el arma del crimen es el Candelabro |
| `arma_cuchillo` | el arma del crimen es el Cuchillo |
| `arma_tuberia` | el arma del crimen es la Tubería de plomo |
| `arma_revolver` | el arma del crimen es el Revólver |
| `lugar_cocina` | el crimen ocurrió en la Cocina |
| `lugar_estudio` | el crimen ocurrió en el Estudio |
| `lugar_biblioteca` | el crimen ocurrió en la Biblioteca |
| `lugar_comedor` | el crimen ocurrió en el Comedor |

#### Grupo 3: Categoría identificada (deducidas)

| `symbol` | `meaning` |
|---|---|
| `sospechoso_identificado` | ya se sabe quién es el culpable |
| `arma_identificada` | ya se sabe cuál es el arma |
| `lugar_identificado` | ya se sabe cuál es el lugar |

#### Grupo 4: Meta (deducida)

| `symbol` | `meaning` |
|---|---|
| `puedo_acusar` | ya puedo hacer la acusación final |

### 2.5 Reglas (25)

Todas son cláusulas de Horn: varias condiciones unidas por `∧` (y) y **una sola conclusión**. Siguen **3 patrones**:

| Grupo | Reglas | Cantidad | Idea |
|---|---|---|---|
| A. Eliminación | R1 a R12 | 12 | Si ya vi las otras tres cartas de una categoría, la cuarta es la solución |
| B. Identificación | R13 a R24 | 12 | Si conozco la solución de una categoría, esa categoría queda resuelta |
| C. Meta | R25 | 1 | Si las tres categorías están resueltas, puedo acusar |

**¿Por qué hay una regla por cada carta?** Una cláusula de Horn no permite "o". No podemos escribir "si el culpable es X o Y o Z...". Por eso cada carta tiene su propia regla, y por eso existe el grupo B: sin esa capa intermedia, `puedo_acusar` necesitaría 4 × 4 × 4 = 64 reglas.

#### Grupo A: Eliminación (R1 a R12)

| Id | Si (antecedentes) | Entonces | En palabras |
|---|---|---|---|
| R1 | `vista_escarlata` ∧ `vista_blanco` ∧ `vista_verde` | `culpable_mostaza` | Vi a los otros tres, así que es Mostaza |
| R2 | `vista_mostaza` ∧ `vista_blanco` ∧ `vista_verde` | `culpable_escarlata` | Vi a los otros tres, así que es Escarlata |
| R3 | `vista_mostaza` ∧ `vista_escarlata` ∧ `vista_verde` | `culpable_blanco` | Vi a los otros tres, así que es Blanco |
| R4 | `vista_mostaza` ∧ `vista_escarlata` ∧ `vista_blanco` | `culpable_verde` | Vi a los otros tres, así que es Verde |
| R5 | `vista_cuchillo` ∧ `vista_tuberia` ∧ `vista_revolver` | `arma_candelabro` | Vi las otras tres armas, así que es el Candelabro |
| R6 | `vista_candelabro` ∧ `vista_tuberia` ∧ `vista_revolver` | `arma_cuchillo` | Vi las otras tres, así que es el Cuchillo |
| R7 | `vista_candelabro` ∧ `vista_cuchillo` ∧ `vista_revolver` | `arma_tuberia` | Vi las otras tres, así que es la Tubería |
| R8 | `vista_candelabro` ∧ `vista_cuchillo` ∧ `vista_tuberia` | `arma_revolver` | Vi las otras tres, así que es el Revólver |
| R9 | `vista_estudio` ∧ `vista_biblioteca` ∧ `vista_comedor` | `lugar_cocina` | Vi los otros tres lugares, así que es la Cocina |
| R10 | `vista_cocina` ∧ `vista_biblioteca` ∧ `vista_comedor` | `lugar_estudio` | Vi los otros tres, así que es el Estudio |
| R11 | `vista_cocina` ∧ `vista_estudio` ∧ `vista_comedor` | `lugar_biblioteca` | Vi los otros tres, así que es la Biblioteca |
| R12 | `vista_cocina` ∧ `vista_estudio` ∧ `vista_biblioteca` | `lugar_comedor` | Vi los otros tres, así que es el Comedor |

#### Grupo B: Identificación de categoría (R13 a R24)

| Id | Si | Entonces | | Id | Si | Entonces |
|---|---|---|---|---|---|---|
| R13 | `culpable_mostaza` | `sospechoso_identificado` | | R19 | `arma_tuberia` | `arma_identificada` |
| R14 | `culpable_escarlata` | `sospechoso_identificado` | | R20 | `arma_revolver` | `arma_identificada` |
| R15 | `culpable_blanco` | `sospechoso_identificado` | | R21 | `lugar_cocina` | `lugar_identificado` |
| R16 | `culpable_verde` | `sospechoso_identificado` | | R22 | `lugar_estudio` | `lugar_identificado` |
| R17 | `arma_candelabro` | `arma_identificada` | | R23 | `lugar_biblioteca` | `lugar_identificado` |
| R18 | `arma_cuchillo` | `arma_identificada` | | R24 | `lugar_comedor` | `lugar_identificado` |

#### Grupo C: Meta (R25)

| Id | Si | Entonces |
|---|---|---|
| R25 | `sospechoso_identificado` ∧ `arma_identificada` ∧ `lugar_identificado` | `puedo_acusar` |

#### Ejemplo de lectura de una regla

**R3:** si la carta del Coronel Mostaza está en una mano conocida, y la de la Señorita Escarlata está en una mano conocida, y la del Reverendo Verde está en una mano conocida, entonces el culpable es la Señora Blanco.

### 2.6 Cadena de razonamiento

Las reglas se encadenan en **3 niveles**: lo que se observa produce la solución, la solución resuelve la categoría y las tres categorías permiten acusar.

| Nivel | Qué contiene | Reglas que lo producen |
|---|---|---|
| 0. Entrada | `vista_*` (dadas por el usuario) | (ninguna) |
| 1. Solución | `culpable_*`, `arma_*`, `lugar_*` | R1 a R12 |
| 2. Categoría | `sospechoso_identificado`, `arma_identificada`, `lugar_identificado` | R13 a R24 |
| 3. Meta | `puedo_acusar` | R25 |

Ejemplo del camino completo para el culpable:

`vista_mostaza ∧ vista_escarlata ∧ vista_verde` → (R3) → `culpable_blanco` → (R15) → `sospechoso_identificado` → (R25, junto con arma y lugar) → `puedo_acusar`

## 3. Implementación

### 3.1 Motor de inferencia

Para este proyecto reutilizamos el **motor de inferencia desarrollado en el laboratorio 05**. Sus componentes son:

| Componente | Qué hace |
|---|---|
| `TValue` | Los tres valores posibles: `YES`, `NO` y `UNKNOWN` |
| `TAtom` | Una proposición atómica (`symbol` y `meaning`) |
| `TRule` | Una cláusula definida de Horn (id, antecedentes y consecuente) |
| `TKnowledgeBase` | La base de conocimiento: proposiciones y reglas, con validación |
| `TFactBase` | La base de hechos: cada hecho con su valor y su origen |
| `applies` | Decide si una regla es aplicable (`YES`, `NO` o `UNKNOWN`) |
| `forward` | Inferencia hacia adelante: deduce todo lo posible desde los hechos |
| `ask`, `deduce`, `backward` | Inferencia hacia atrás: demuestra una meta y pregunta al usuario lo que falte |
| `explain` | Explica el "cómo" de una conclusión recorriendo los orígenes en la FB |
| `DevelopmentSubsystem` | Subsistema de desarrollo: permite al experto construir la KB |
| `QuerySubsystem` | Subsistema de consulta: permite al usuario deducir, responder y explicar |

Fuente: laboratorio 05 de IAIA.

In [ ]:
class TValue:
    """Los valores para una proposición atómica."""

    yes = "YES"
    no = "NO"
    unknown = "UNKNOWN"

In [ ]:
class TAtom:
    """Una proposición atómica, con su variable y significado."""

    def __init__(self, symbol: str, meaning: str) -> None:
        self.symbol = symbol
        self.meaning = meaning

    def __eq__(self, other):
        return self.symbol == other.symbol and self.meaning == other.meaning

    def __hash__(self):
        return hash((self.symbol, self.meaning))

    def __str__(self) -> str:
        return self.symbol

In [ ]:
class TRule:
    """Una cláusula definida."""

    def __init__(self, name: str, antecedents: list[TAtom], consequent: TAtom) -> None:
        self.name = name
        self.antecedents = antecedents
        self.consequent = consequent

    def __str__(self) -> str:
        """Representación en lenguaje natural de la regla."""
        antecedents_str = " y ".join(map(str, self.antecedents))
        return f"{self.name}: {antecedents_str}, entonces {self.consequent}"

In [ ]:
class TKnowledgeBase:
    """La base de conocimiento: proposiciones y cláusulas definidas.

    Almacena dos conjuntos del dominio: atoms, las proposiciones (TAtom),
    y rules, las reglas (TRule), que se identifican por su name.
    """

    def __init__(self) -> None:
        self.atoms = {}
        self.rules = {}

    def add(self, item: TAtom | TRule) -> None:
        """Agregar una proposición o una regla.

        Al agregar una regla, sus antecedents y su consequent ya deben
        estar registrados en atoms.

        :param item: La proposición o la regla a agregar.
        :raises ValueError: Si la regla usa proposiciones no registradas.
        """
        if isinstance(item, TAtom):
            self.atoms[item] = item
        elif isinstance(item, TRule):
            for atom in item.antecedents:
                if atom not in self.atoms:
                    raise ValueError(f"Antecedente {atom} no registrado")
            if item.consequent not in self.atoms:
                raise ValueError(f"Consecuente {item.consequent} no registrado")
            self.rules[item.name] = item

    def obtain(self, rule_name: str) -> TRule:
        """Recuperar una regla por su identificador."""
        return self.rules[rule_name]

    def has_rule(self, rule_name: str) -> bool:
        """Conocer si un identificador de regla ya está registrado."""
        return rule_name in self.rules

    def atom(self, symbol: str) -> TAtom | None:
        """Recuperar una proposición por su symbol (None si no existe)."""
        for atom in self.atoms.values():
            if atom.symbol == symbol:
                return atom
        return None

In [ ]:
class TFactBase:
    """La base de hechos: proposiciones atómicas con su valor y origen.

    Guarda en facts, para cada atom (la clave), su value y su source:
    'user' si lo dio el usuario, o el identificador de la regla que lo dedujo.
    """

    def __init__(self) -> None:
        self.facts = {}

    def set(self, atom: TAtom, value: TValue, source: str) -> None:
        """Agregar o actualizar un hecho."""
        self.facts[atom] = (value, source)

    def value(self, atom: TAtom) -> TValue:
        """Conocer el valor de un hecho."""
        return self.facts[atom][0]

    def source(self, atom: TAtom) -> str | None:
        """Conocer el origen de un hecho (None si no está en la FB)."""
        return self.facts[atom][1] if atom in self.facts else None

In [ ]:
def applies(kb: TKnowledgeBase, fb: TFactBase, rule: TRule) -> TValue:
    """Conocer si una regla es aplicable a la base de hechos dada.

    :return: YES si la regla es aplicable ahora;
             NO si ya no podrá aplicarse;
             UNKNOWN si falta información.
    """
    has_unknown = False
    for atom in rule.antecedents:
        if atom not in fb.facts:
            has_unknown = True
            continue

        val = fb.value(atom)
        if val == TValue.no:
            return TValue.no
        elif val == TValue.unknown:
            return TValue.no

    if has_unknown:
        return TValue.unknown
    return TValue.yes

In [ ]:
def forward(kb: TKnowledgeBase, fb: TFactBase) -> TFactBase:
    """Inferencia con razonamiento hacia adelante.

    :return: La base de hechos actualizada con los hechos inferidos;
        cada hecho nuevo tiene como origen el identificador de su regla.
    """
    changed = True
    while changed:
        changed = False
        for rule in kb.rules.values():
            if applies(kb, fb, rule) == TValue.yes:
                if rule.consequent not in fb.facts or fb.value(rule.consequent) != TValue.yes:
                    fb.set(rule.consequent, TValue.yes, rule.name)
                    changed = True
    return fb

In [ ]:
def ask(p: TAtom) -> TValue:
    """Preguntar al usuario por el valor de una proposición atómica.

    Muestra el meaning de p y acepta YES, NO o UNKNOWN; insiste hasta
    recibir una respuesta válida.
    """
    while True:
        print(f"¿Es verdadero que: {p.meaning}? ({p.symbol})")
        respuesta = input("Ingrese su respuesta (YES / NO / UNKNOWN): ").strip().upper()
        if respuesta == "YES":
            return TValue.yes
        elif respuesta == "NO":
            return TValue.no
        elif respuesta == "UNKNOWN":
            return TValue.unknown
        else:
            print("Respuesta incorrecta.\n")

In [ ]:
def deduce(kb: TKnowledgeBase, p: TAtom) -> list[TRule]:
    """Conocer las reglas que permiten concluir una proposición atómica.

    :return: Las reglas cuyo consecuente es p.
    """
    return [rule for rule in kb.rules.values() if rule.consequent == p]

In [ ]:
def backward(
    kb: TKnowledgeBase, fb: TFactBase, p: TAtom
) -> tuple[TValue, TFactBase]:
    """Inferencia con razonamiento hacia atrás.

    :return: El valor de p y la base de hechos actualizada.
    """
    if p in fb.facts:
        return fb.value(p), fb

    rules = deduce(kb, p)
    for rule in rules:
        all_antecedents_yes = True
        for antecedent in rule.antecedents:
            val_antecedent, fb = backward(kb, fb, antecedent)
            if val_antecedent != TValue.yes:
                all_antecedents_yes = False
                break

        if all_antecedents_yes:
            fb.set(p, TValue.yes, rule.name)
            return TValue.yes, fb

    if not rules:
        user_val = ask(p)
        fb.set(p, user_val, "user")
        return user_val, fb
    else:
        fb.set(p, TValue.unknown, "system")
        return TValue.unknown, fb

In [ ]:
def explain(kb: TKnowledgeBase, fb: TFactBase, p: TAtom, depth: int = 0) -> str:
    """Explicar el cómo de una conclusión recorriendo los orígenes en la FB.

    :return: La cadena de justificación, con un nivel de sangría por regla.
    """
    indent = "  " * depth
    if p not in fb.facts:
        return f"{indent}- {p.symbol}: No está registrado en la base de hechos.\n"

    val = fb.value(p)
    src = fb.source(p)

    if src == "user":
        return f"{indent}- {p.meaning} ({p.symbol}) = {val} porque lo indicó el usuario.\n"
    elif src == "system":
        return f"{indent}- {p.meaning} ({p.symbol}) = {val} por descarte del sistema.\n"
    elif src is not None and kb.has_rule(src):
        rule = kb.obtain(src)
        ans = f"{indent}- {p.meaning} ({p.symbol}) = {val} por la regla {rule.name}: "
        ans += " y ".join([ant.symbol for ant in rule.antecedents]) + f" => {rule.consequent.symbol}\n"
        for antecedent in rule.antecedents:
            ans += explain(kb, fb, antecedent, depth + 1)
        return ans
    else:
        return f"{indent}- {p.meaning} ({p.symbol}) = {val} (origen desconocido: {src})\n"

In [ ]:
class DevelopmentSubsystem:
    """Subsistema de desarrollo: construir el conocimiento."""

    def __init__(self, kb: TKnowledgeBase) -> None:
        self.kb = kb

    def add_atom(self) -> None:
        """Pedir symbol y meaning y registrar la proposición en la KB."""
        symbol = input("Symbol (sin espacios, ej. tiene_pelo): ").strip()
        meaning = input("Meaning (ej. el animal tiene pelo): ").strip()

        if not symbol or not meaning or " " in symbol:
            print("Symbol y meaning son obligatorios, y symbol no lleva espacios.")
            return
        if self.kb.atom(symbol) is not None:
            print(f"Ya existe una proposición con symbol '{symbol}'.")
            return

        self.kb.add(TAtom(symbol, meaning))
        print(f"Proposición '{symbol}' registrada.")

    def add_rule(self) -> None:
        """Pedir name, antecedents y consequent y registrar la regla."""
        name = input("Identificador de la regla (ej. R1): ").strip()
        if not name:
            print("El identificador es obligatorio.")
            return
        if self.kb.has_rule(name):
            print(f"Ya existe una regla '{name}'.")
            return

        raw = input("Antecedentes (symbols separados por coma): ")
        symbols = [s.strip() for s in raw.split(",") if s.strip()]
        if not symbols:
            print("Debe haber al menos un antecedente.")
            return

        antecedents = []
        for s in symbols:
            atom = self.kb.atom(s)
            if atom is None:
                print(f"La proposición '{s}' no está registrada. Agrégala primero.")
                return
            antecedents.append(atom)

        cons_symbol = input("Consecuente (symbol): ").strip()
        consequent = self.kb.atom(cons_symbol)
        if consequent is None:
            print(f"La proposición '{cons_symbol}' no está registrada. Agrégala primero.")
            return

        try:
            self.kb.add(TRule(name, antecedents, consequent))
            print(f"Regla '{name}' registrada.")
        except ValueError as e:
            print(f"No se pudo registrar la regla: {e}")

    def show(self) -> None:
        """Mostrar las proposiciones y reglas registradas."""
        print("PROPOSICIONES")
        if not self.kb.atoms:
            print("(ninguna)")
        for a in self.kb.atoms.values():
            print(f"- {a.symbol}: {a.meaning}")

        print("REGLAS")
        if not self.kb.rules:
            print("(ninguna)")
        for r in self.kb.rules.values():
            ants = " ∧ ".join(a.meaning for a in r.antecedents)
            print(f"{r.name}: {ants} ⇒ {r.consequent.meaning}")
        print()

    def reset(self) -> None:
        """Iniciar un nuevo conocimiento (vaciar la KB)."""
        self.kb.atoms.clear()
        self.kb.rules.clear()
        print("KB vaciada.")

    def run(self) -> None:
        """Mostrar el menú y ejecutar comandos hasta que el usuario salga."""
        comandos = {
            "1": self.add_atom,
            "2": self.add_rule,
            "3": self.show,
            "4": self.reset,
        }
        while True:
            print("Subsistema de desarrollo")
            print("1. Agregar proposición")
            print("2. Agregar regla")
            print("3. Mostrar KB")
            print("4. Reiniciar KB")
            print("0. Salir")
            opcion = input("Opción: ").strip()
            if opcion == "0":
                break
            accion = comandos.get(opcion)
            if accion:
                accion()
            else:
                print("Opción inválida.")

In [ ]:
class QuerySubsystem:
    """Subsistema de consulta: usar el conocimiento."""

    def __init__(self, kb: TKnowledgeBase, fb: TFactBase) -> None:
        self.kb = kb
        self.fb = fb

    def pedir_atom(self) -> TAtom | None:
        symbol = input("Symbol de la proposición: ").strip()
        atom = self.kb.atom(symbol)
        if atom is None:
            print("Esa proposición no existe en la KB.")
        return atom

    def add_fact(self) -> None:
        """Pedir symbol y valor y registrar el hecho en la FB."""
        atom = self.pedir_atom()
        if atom is None:
            return
        value = ask(atom)
        self.fb.set(atom, value, "user")
        print("Hecho registrado.")

    def infer(self) -> None:
        """Ejecutar forward y mostrar los hechos recién deducidos."""
        antes = dict(self.fb.facts)
        forward(self.kb, self.fb)
        nuevos = 0
        for atom, fact in self.fb.facts.items():
            if antes.get(atom) != fact:
                print(f"{atom.meaning} = {fact[0]}, origen {fact[1]}")
                nuevos += 1
        if nuevos == 0:
            print("No se dedujo nada nuevo.")

    def answer(self) -> None:
        """Pedir una proposición y responderla con backward."""
        atom = self.pedir_atom()
        if atom is None:
            return
        value, self.fb = backward(self.kb, self.fb, atom)
        print(f"{atom.meaning} = {value}, origen {self.fb.source(atom)}")

    def how(self) -> None:
        """Pedir una conclusión y mostrar su explicación con explain."""
        atom = self.pedir_atom()
        if atom is None:
            return
        print(explain(self.kb, self.fb, atom))

    def show(self) -> None:
        """Mostrar la FB actual."""
        if not self.fb.facts:
            print("La FB está vacía.")
        for atom, fact in self.fb.facts.items():
            print(f"{atom.symbol} = {fact[0]}, origen {fact[1]}")

    def reset(self) -> None:
        """Iniciar una nueva consulta (vaciar la FB sin tocar la KB)."""
        self.fb.facts.clear()
        print("FB vaciada.")

    def run(self) -> None:
        """Mostrar el menú y ejecutar comandos hasta que el usuario salga."""
        comandos = {
            "1": self.add_fact,
            "2": self.infer,
            "3": self.answer,
            "4": self.how,
            "5": self.show,
            "6": self.reset,
        }
        while True:
            print("1. Agregar hecho")
            print("2. Inferir con forward")
            print("3. Responder con backward")
            print("4. Explicar el cómo")
            print("5. Mostrar FB")
            print("6. Nueva consulta")
            print("0. Salir")
            opcion = input("Opción: ").strip()
            if opcion == "0":
                break
            if opcion in comandos:
                comandos[opcion]()
            else:
                print("Opción inválida.")

### 3.2 Construcción de la base de conocimiento

La KB se construye con el **Subsistema de Desarrollo** del motor (`DevelopmentSubsystem`), el mismo que usaría un experto para ingresar conocimiento: primero se registran las proposiciones y luego las reglas.

Como las 28 proposiciones y las 25 reglas siguen patrones fijos, las **generamos con ciclos** a partir de los datos del dominio (las 12 cartas) en lugar de escribirlas una por una. Esto evita errores de digitación y asegura que los nombres sean consistentes.

| Paso | Qué hace |
|---|---|
| 1 | Define los datos del dominio: las cartas de cada categoría |
| 2 | Genera la lista de 28 proposiciones (`symbol` y `meaning`) |
| 3 | Genera la lista de 25 reglas (id, antecedentes, consecuente) |
| 4 | `build_kb()` registra todo en la KB usando el Subsistema de Desarrollo |

El motor **valida** cada regla al registrarla: si usa una proposición que no está declarada, la rechaza con un error.


In [ ]:
import io
from contextlib import redirect_stdout
from unittest.mock import patch


def con_entradas(respuestas, accion):
    with patch("builtins.input", side_effect=respuestas):
        accion()

In [ ]:
CARTAS = {
    "sospechoso": [
        ("mostaza", "el Coronel Mostaza", "del Coronel Mostaza"),
        ("escarlata", "la Señorita Escarlata", "de la Señorita Escarlata"),
        ("blanco", "la Señora Blanco", "de la Señora Blanco"),
        ("verde", "el Reverendo Verde", "del Reverendo Verde"),
    ],
    "arma": [
        ("candelabro", "el Candelabro", "del Candelabro"),
        ("cuchillo", "el Cuchillo", "del Cuchillo"),
        ("tuberia", "la Tubería de plomo", "de la Tubería de plomo"),
        ("revolver", "el Revólver", "del Revólver"),
    ],
    "lugar": [
        ("cocina", "la Cocina", "de la Cocina"),
        ("estudio", "el Estudio", "del Estudio"),
        ("biblioteca", "la Biblioteca", "de la Biblioteca"),
        ("comedor", "el Comedor", "del Comedor"),
    ],
}

CATEGORIAS = {
    "sospechoso": {
        "prefijo": "culpable_",
        "frase": "el culpable es {}",
        "resuelta": ["sospechoso_identificado", "ya se sabe quién es el culpable"],
    },
    "arma": {
        "prefijo": "arma_",
        "frase": "el arma del crimen es {}",
        "resuelta": ["arma_identificada", "ya se sabe cuál es el arma"],
    },
    "lugar": {
        "prefijo": "lugar_",
        "frase": "el crimen ocurrió en {}",
        "resuelta": ["lugar_identificado", "ya se sabe cuál es el lugar"],
    },
}

In [ ]:
ATOMOS = []

#cartas vistas
for cat, cartas in CARTAS.items():
    for clave, _, con_de in cartas:
        ATOMOS.append([f"vista_{clave}", f"la carta {con_de} está en una mano conocida"])

#solucion (deducidas)
for cat, cartas in CARTAS.items():
    for clave, con_art, _ in cartas:
        pref = CATEGORIAS[cat]["prefijo"]
        ATOMOS.append([f"{pref}{clave}", CATEGORIAS[cat]["frase"].format(con_art)])

#categoria resuelta (deducidas)
for cat in CATEGORIAS:
    ATOMOS.append(CATEGORIAS[cat]["resuelta"])

#meta
ATOMOS.append(["puedo_acusar", "ya puedo hacer la acusación final"])

print(len(ATOMOS), "proposiciones")

28 proposiciones


In [ ]:
REGLAS = []
n = 1

# Grupo A: eliminación (R1 a R12)
for cat, cartas in CARTAS.items():
    pref = CATEGORIAS[cat]["prefijo"]
    for i, (clave, _, _) in enumerate(cartas):
        otras = [f"vista_{c[0]}" for j, c in enumerate(cartas) if j != i]
        REGLAS.append([f"R{n}", otras, f"{pref}{clave}"])
        n += 1

# Grupo B: identificación de categoría (R13 a R24)
for cat, cartas in CARTAS.items():
    pref = CATEGORIAS[cat]["prefijo"]
    resuelta = CATEGORIAS[cat]["resuelta"][0]
    for clave, _, _ in cartas:
        REGLAS.append([f"R{n}", [f"{pref}{clave}"], resuelta])
        n += 1

# Grupo C: meta (R25)
REGLAS.append([
    f"R{n}",
    ["sospechoso_identificado", "arma_identificada", "lugar_identificado"],
    "puedo_acusar",
])

print(len(REGLAS), "reglas")

25 reglas


In [ ]:
def build_kb():
    kb = TKnowledgeBase()
    dev = DevelopmentSubsystem(kb)
    with redirect_stdout(io.StringIO()):
        for datos in ATOMOS:
            con_entradas(datos, dev.add_atom)
        for name, antecedentes, consecuente in REGLAS:
            con_entradas([name, ", ".join(antecedentes), consecuente], dev.add_rule)
    return kb

In [ ]:
kb = build_kb()
DevelopmentSubsystem(kb).show()

PROPOSICIONES
- vista_mostaza: la carta del Coronel Mostaza está en una mano conocida
- vista_escarlata: la carta de la Señorita Escarlata está en una mano conocida
- vista_blanco: la carta de la Señora Blanco está en una mano conocida
- vista_verde: la carta del Reverendo Verde está en una mano conocida
- vista_candelabro: la carta del Candelabro está en una mano conocida
- vista_cuchillo: la carta del Cuchillo está en una mano conocida
- vista_tuberia: la carta de la Tubería de plomo está en una mano conocida
- vista_revolver: la carta del Revólver está en una mano conocida
- vista_cocina: la carta de la Cocina está en una mano conocida
- vista_estudio: la carta del Estudio está en una mano conocida
- vista_biblioteca: la carta de la Biblioteca está en una mano conocida
- vista_comedor: la carta del Comedor está en una mano conocida
- culpable_mostaza: el culpable es el Coronel Mostaza
- culpable_escarlata: el culpable es la Señorita Escarlata
- culpable_blanco: el culpable es la Señ

### Verificación de la KB

Antes de continuar comprobamos que la KB quedó bien construida:

| Prueba | Resultado esperado |
|---|---|
| Número de proposiciones | 28 |
| Número de reglas | 25 |
| Recuperar una regla por su identificador (R3, R25) | Se muestra la regla |
| Buscar una proposición por su `symbol` | Se recupera con su `meaning` |
| Registrar una regla con una proposición no declarada | Error `ValueError` |

In [ ]:
print("Proposiciones:", len(kb.atoms))
print("Reglas:", len(kb.rules))
assert len(kb.atoms) == 28 and len(kb.rules) == 25

print(kb.obtain("R3"))
print(kb.obtain("R25"))
print(kb.atom("vista_tuberia").meaning)

no_registrada = TAtom("carta_extra", "una carta que no existe")
try:
    kb.add(TRule("R_X", [no_registrada], kb.atom("puedo_acusar")))
    print("Error: se permitió una regla con proposición no registrada")
except ValueError as e:
    print("Error esperado:", e)

Proposiciones: 28
Reglas: 25
R3: vista_mostaza y vista_escarlata y vista_verde, entonces culpable_blanco
R25: sospechoso_identificado y arma_identificada y lugar_identificado, entonces puedo_acusar
la carta de la Tubería de plomo está en una mano conocida
Error esperado: Antecedente carta_extra no registrado


## 4. Escenarios de prueba (bases de hechos)

Una **base de hechos** representa lo que el jugador sabe en un momento de la partida. Cada hecho guarda la proposición, su valor y su origen (`user` si lo dio el usuario, o el id de la regla que lo dedujo). Probamos **dos escenarios** distintos.

> **Nota:** el "sobre" solo sirve para que nosotros sepamos la respuesta correcta y podamos comprobar el resultado. El sistema **no lo conoce**: razona únicamente con lo que hay en la base de hechos.

| | Escenario A: "Caso resuelto" | Escenario B: "Falta un dato" |
|---|---|---|
| **Qué demuestra** | Forward chaining completo y explicación | Forward parcial, backward con pregunta al usuario y manejo de `UNKNOWN` |
| **Sobre (solución oculta)** | Señora Blanco, Tubería de plomo, Biblioteca | Señorita Escarlata, Revólver, Estudio |
| **Hechos `YES`** | 9: `vista_mostaza`, `vista_escarlata`, `vista_verde`, `vista_candelabro`, `vista_cuchillo`, `vista_revolver`, `vista_cocina`, `vista_estudio`, `vista_comedor` | 8: `vista_mostaza`, `vista_blanco`, `vista_verde`, `vista_candelabro`, `vista_cuchillo`, `vista_cocina`, `vista_biblioteca`, `vista_comedor` |
| **Hechos `UNKNOWN`** | (ninguno) | 3: `vista_escarlata`, `vista_revolver`, `vista_estudio` (cartas que nadie me ha mostrado) |
| **Ausente de la FB** | (ninguno) | `vista_tuberia` (el dato que el sistema deberá preguntar) |
| **Resultado esperado** | `puedo_acusar = YES` | B1: `YES`. B2: `UNKNOWN` |

**¿Por qué en B las cartas no mostradas son `UNKNOWN` y no están ausentes?** Porque si estuvieran ausentes, el backward también las preguntaría y la demostración se llenaría de preguntas irrelevantes. Con `UNKNOWN` el sistema entiende que ya se intentó y no insiste.

In [ ]:
def build_fb_a(kb):
    fb = TFactBase()
    for s in ["vista_mostaza", "vista_escarlata", "vista_verde",
              "vista_candelabro", "vista_cuchillo", "vista_revolver",
              "vista_cocina", "vista_estudio", "vista_comedor"]:
        fb.set(kb.atom(s), TValue.yes, "user")
    return fb


def build_fb_b(kb):
    fb = TFactBase()
    for s in ["vista_mostaza", "vista_blanco", "vista_verde",
              "vista_candelabro", "vista_cuchillo",
              "vista_cocina", "vista_biblioteca", "vista_comedor"]:
        fb.set(kb.atom(s), TValue.yes, "user")
    for s in ["vista_escarlata", "vista_revolver", "vista_estudio"]:
        fb.set(kb.atom(s), TValue.unknown, "user")
    # vista_tuberia NO se registra: es el dato que el backward preguntará
    return fb

In [ ]:
print("Escenario A (9 hechos):")
QuerySubsystem(kb, build_fb_a(kb)).show()

print("\nEscenario B (8 YES, 3 UNKNOWN, vista_tuberia ausente):")
QuerySubsystem(kb, build_fb_b(kb)).show()

Escenario A (9 hechos):
vista_mostaza = YES, origen user
vista_escarlata = YES, origen user
vista_verde = YES, origen user
vista_candelabro = YES, origen user
vista_cuchillo = YES, origen user
vista_revolver = YES, origen user
vista_cocina = YES, origen user
vista_estudio = YES, origen user
vista_comedor = YES, origen user

Escenario B (8 YES, 3 UNKNOWN, vista_tuberia ausente):
vista_mostaza = YES, origen user
vista_blanco = YES, origen user
vista_verde = YES, origen user
vista_candelabro = YES, origen user
vista_cuchillo = YES, origen user
vista_cocina = YES, origen user
vista_biblioteca = YES, origen user
vista_comedor = YES, origen user
vista_escarlata = UNKNOWN, origen user
vista_revolver = UNKNOWN, origen user
vista_estudio = UNKNOWN, origen user
